# RIGHTWARE Jarvis do Issa — GPU Kaggle, repositório completo, sem token

Clona **rightware-corporations/jarvis-v2.0** na branch develop com submódulos, instala a revisão fixada e testa os modelos Qwen3.5 2B/4B com o **wrapper RIGHTWARE real**. O repositório tem de estar **PÚBLICO**, após revisão de segredos e histórico, para clonar sem token. Não fornecer tokens. Activar GPU e Internet. Guardar o ZIP de evidências.


In [ ]:
from pathlib import Path
import os, subprocess, sys, sysconfig

ROOT = Path("/kaggle/working")
REPO = ROOT / "jarvis-v2.0"
URL = "https://github.com/rightware-corporations/jarvis-v2.0.git"

# This is the FULL RIGHTWARE wrapper test, not an upstream-only comparison.
# Only run after the owner makes the repository PUBLIC and reviews its history.
# Public GitHub cloning needs no personal token.
if REPO.exists() and not (REPO / ".git").exists():
    raise RuntimeError(f"{REPO} existe mas não é um checkout Git válido.")
if not REPO.exists():
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    result = subprocess.run(["git", "clone", "--branch", "develop",
                             "--recurse-submodules", URL, str(REPO)],
                            text=True, capture_output=True, env=env)
    if result.returncode:
        raise RuntimeError(
            "Clone sem token falhou. Confirma que o repo jarvis-v2.0 "
            "está PUBLIC e acessível; não partilhes credenciais. "
            + result.stderr[-350:]
        )
else:
    print("CHECKOUT EXISTENTE: não actualizado automaticamente; verificar SHA.")
subprocess.run(["git", "status", "--short"], cwd=REPO, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, check=True)
subprocess.run(["git", "submodule", "status"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv", "zstandard"], check=True)
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ.get("PATH", "")
subprocess.run([sys.executable, "jarvis.py", "setup"], cwd=REPO, check=True)
print("CHECKOUT + FROZEN DEPENDENCIES: ready")


In [ ]:
import hashlib, io, json, os, subprocess, tarfile, time, urllib.request
import zstandard
from pathlib import Path

OLLAMA_VERSION = "0.40.15"
ARCHIVE_URL = f"https://github.com/ollama/ollama/releases/download/v{OLLAMA_VERSION}/ollama-linux-amd64.tar.zst"
EXPECTED_SHA256 = "628c018404a4206493efd38e46251c154ede5141b8b8dc85c659b3b027c47379"
OLLAMA_ROOT = ROOT / ".ollama-runtime"
OLLAMA_BIN = OLLAMA_ROOT / "bin" / "ollama"
MODELS_DIR = ROOT / ".ollama-models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
def api(path):
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    with opener.open("http://127.0.0.1:11434" + path, timeout=5) as response:
        return json.load(response)
try:
    existing = api("/api/version")
except Exception:
    existing = None
if not existing:
    if not OLLAMA_BIN.is_file():
        archive = ROOT / f"ollama-linux-amd64-v{OLLAMA_VERSION}.tar.zst"
        if not archive.exists():
            print("A descarregar o Ollama oficial; ficheiro grande...")
            urllib.request.urlretrieve(ARCHIVE_URL, archive)
        digest = hashlib.file_digest(archive.open("rb"), "sha256").hexdigest()
        if digest != EXPECTED_SHA256:
            raise RuntimeError("Ollama archive SHA256 inválido. Não extrair.")
        OLLAMA_ROOT.mkdir(parents=True, exist_ok=True)
        with archive.open("rb") as file, zstandard.ZstdDecompressor().stream_reader(file) as reader:
            with tarfile.open(fileobj=reader, mode="r|") as tar:
                tar.extractall(OLLAMA_ROOT, filter="data")
    env = dict(os.environ, OLLAMA_NO_CLOUD="1", OLLAMA_HOST="127.0.0.1:11434",
               OLLAMA_MODELS=str(MODELS_DIR), CUDA_VISIBLE_DEVICES="0")
    log = (ROOT / "ollama_server.log").open("a")
    ollama_proc = subprocess.Popen([str(OLLAMA_BIN), "serve"], env=env,
                                    stdout=log, stderr=subprocess.STDOUT)
    for _ in range(90):
        try:
            existing = api("/api/version")
            break
        except Exception:
            time.sleep(2)
    if not existing:
        raise RuntimeError("Ollama não iniciou. Consultar ollama_server.log.")
else:
    print("Ollama já activo: cloud/variáveis do servidor existente NÃO verificadas.")
if existing.get("version") != OLLAMA_VERSION:
    raise RuntimeError(f"Ollama {existing['version']}; esperado {OLLAMA_VERSION}. Não comparar versões diferentes.")
os.environ.update({"OLLAMA_NO_CLOUD":"1", "OLLAMA_HOST":"127.0.0.1:11434",
                   "OLLAMA_MODELS":str(MODELS_DIR), "CUDA_VISIBLE_DEVICES":"0"})
binary = str(OLLAMA_BIN) if OLLAMA_BIN.is_file() else "ollama"
for model in ("qwen3.5:2b", "qwen3.5:4b"):
    print("MODEL:", model, flush=True)
    subprocess.run([binary, "pull", model], check=True, env=os.environ)
print("OLLAMA + MODELS READY", existing)


In [ ]:
import datetime, os, subprocess, zipfile
from pathlib import Path
from IPython.display import FileLink, display

PYTHON = REPO / "vendor" / "openjarvis" / ".venv" / "bin" / "python"
EVIDENCE = ROOT / "jarvis_issa_evidence"
EVIDENCE.mkdir(exist_ok=True)
report = EVIDENCE / "comparison_2b_4b.json"
cmd = [str(PYTHON), "-m", "qa.compare_models",
       "--models", "qwen3.5:2b", "qwen3.5:4b",
       "--output", str(report)]
print("RUN:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO, check=True, env=os.environ)
sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
(EVIDENCE / "README.txt").write_text(
    "RIGHTWARE Jarvis do Issa — synthetic real-model comparison.\n"
    f"repo develop SHA: {sha}\n"
    "No automatic quality score. Inspect answers and calculator tool_results.\n"
    "The Kaggle runtime is temporary; this ZIP must be preserved externally.\n",
    encoding="utf-8"
)
zip_path = ROOT / ("JARVIS_ISSA_QWEN_EVIDENCE_" +
                   datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%SZ") + ".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in EVIDENCE.iterdir():
        if file.is_file():
            archive.write(file, arcname=file.name)
print("ZIP EVIDENCE:", zip_path)
display(FileLink(str(zip_path)))
